# 📊 Data Exploration: Across the Homophily Spectrum

This notebook explores our three carefully chosen datasets:
1. **Amazon Computers** — High homophily (~0.78)
2. **WikiCS** — Medium homophily (~0.65)
3. **Roman-empire** — Very low homophily / Heterophilic (~0.05)

We examine node features, edge semantics, class distributions, degree distributions, and homophily characteristics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch_geometric.utils import degree

from src.datasets import (
    load_dataset, get_all_dataset_names, 
    print_dataset_summary, get_degree_distribution,
    DATASET_INFO
)
from src.utils import compute_homophily, compute_node_homophily, set_seed

set_seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print("✅ All imports successful!")

## 1. Load All Datasets

In [ ]:
# Load all three datasets
datasets = {}
for name in get_all_dataset_names():
    print(f"Loading {name}...")
    datasets[name] = load_dataset(name, root='../data')
    print(f"  ✅ Loaded!\n")

print(f"\nLoaded {len(datasets)} datasets: {list(datasets.keys())}")

## 2. Dataset Summaries

In [ ]:
# Print comprehensive summary for each dataset
for name, data in datasets.items():
    print_dataset_summary(data, name)
    print()

## 3. Comparative Statistics Table

In [ ]:
import pandas as pd

stats = []
for name, data in datasets.items():
    deg = degree(data.edge_index[0], num_nodes=data.num_nodes)
    h = compute_homophily(data.edge_index, data.y)
    
    stats.append({
        'Dataset': data.dataset_name,
        'Nodes': data.num_nodes,
        'Edges': data.num_edges,
        'Features': data.num_node_features,
        'Classes': data.num_classes,
        'Avg Degree': f"{deg.mean().item():.1f}",
        'Edge Homophily': f"{h:.4f}",
        'Homophily Level': DATASET_INFO[name]['homophily_level'],
        'Train Nodes': data.train_mask.sum().item(),
        'Val Nodes': data.val_mask.sum().item(),
        'Test Nodes': data.test_mask.sum().item(),
    })

stats_df = pd.DataFrame(stats)
stats_df

## 4. Degree Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, (name, data) in enumerate(datasets.items()):
    degrees = get_degree_distribution(data)
    
    ax = axes[i]
    ax.hist(degrees, bins=50, color=colors[i], alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.set_title(f"{data.dataset_name}\nAvg: {degrees.mean():.1f}, Max: {degrees.max():.0f}", 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Node Degree')
    ax.set_ylabel('Count')
    ax.set_yscale('log')
    
    # Add vertical line for mean
    ax.axvline(degrees.mean(), color='red', linestyle='--', alpha=0.8, label=f'Mean={degrees.mean():.1f}')
    ax.legend()

plt.suptitle('Degree Distributions Across the Homophily Spectrum', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/degree_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/degree_distributions.png")

## 5. Class Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, data) in enumerate(datasets.items()):
    classes, counts = torch.unique(data.y, return_counts=True)
    
    ax = axes[i]
    bars = ax.bar(classes.numpy(), counts.numpy(), color=colors[i], alpha=0.7, edgecolor='white')
    ax.set_title(f"{data.dataset_name}\n({data.num_classes} classes)",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Class Label')
    ax.set_ylabel('Count')
    
    # Annotate imbalance ratio
    imbalance = counts.max().item() / counts.min().item()
    ax.text(0.95, 0.95, f'Imbalance: {imbalance:.1f}x',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Class Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/class_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/class_distributions.png")

## 6. Homophily Analysis

### 6.1 Edge Homophily Comparison

In [ ]:
# Compute homophily for each dataset
homophily_data = []
for name, data in datasets.items():
    h = compute_homophily(data.edge_index, data.y)
    homophily_data.append({
        'dataset': data.dataset_name,
        'homophily': h,
    })

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = [d['dataset'] for d in homophily_data]
values = [d['homophily'] for d in homophily_data]

bars = ax.bar(names, values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)

# Add value labels on bars
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Reference lines
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Boundary (0.5)')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Edge Homophily Ratio', fontsize=12)
ax.set_title('The Homophily Spectrum: Our Three Datasets', fontsize=14, fontweight='bold')
ax.legend()

# Annotations
ax.annotate('← Heterophilic', xy=(0.02, 0.15), fontsize=10, color='red', 
            fontweight='bold', transform=ax.transAxes)
ax.annotate('Homophilic →', xy=(0.82, 0.85), fontsize=10, color='green',
            fontweight='bold', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('../results/plots/homophily_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/homophily_spectrum.png")

### 6.2 Node-Level Homophily Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, data) in enumerate(datasets.items()):
    node_h = compute_node_homophily(data.edge_index, data.y, data.num_nodes)
    
    ax = axes[i]
    ax.hist(node_h, bins=50, color=colors[i], alpha=0.7, edgecolor='white', density=True)
    ax.axvline(node_h.mean(), color='red', linestyle='--', label=f'Mean={node_h.mean():.3f}')
    ax.set_title(f"{data.dataset_name}", fontsize=13, fontweight='bold')
    ax.set_xlabel('Node-Level Homophily')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Node-Level Homophily Distributions\n(Per-node fraction of same-class neighbors)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('../results/plots/node_homophily_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/node_homophily_distributions.png")

## 7. Feature Space Preview (PCA)

Quick PCA projection to see if features alone can separate classes (before GNN).

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, data) in enumerate(datasets.items()):
    pca = PCA(n_components=2, random_state=42)
    x_2d = pca.fit_transform(data.x.numpy())
    
    ax = axes[i]
    scatter = ax.scatter(x_2d[:, 0], x_2d[:, 1], c=data.y.numpy(), 
                         cmap='tab10' if data.num_classes <= 10 else 'tab20',
                         s=3, alpha=0.3)
    ax.set_title(f"{data.dataset_name}\nPCA of Raw Features",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.colorbar(scatter, ax=ax, label='Class')

plt.suptitle('Raw Feature Space (PCA) — Before GNN',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/plots/raw_feature_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/plots/raw_feature_pca.png")

## 8. Summary

### Key Observations to Report in Milestone 1:

1. **Homophily Spectrum**: Our datasets span the full range — Amazon Computers (high, ~0.78), WikiCS (medium, ~0.65), Roman-empire (very low, ~0.05)
2. **Scale**: All datasets are medium-scale (10K-23K nodes), manageable for thorough experiments
3. **Feature Dimensionality**: Varies from 300 (WikiCS, Roman-empire) to 767 (Amazon), testing model robustness to feature space size
4. **Class Count**: Ranges from 10 (WikiCS, Amazon) to 18 (Roman-empire), with varying class balance
5. **Graph Density**: Amazon Computers is much denser (higher avg degree) than the others

These differences will create interesting dynamics when we compare GNN architectures in Milestone 2.